In [3]:
from google.colab import files
import pandas as pd
import io
uploaded = files.upload()

#print(df)
#print(type(df))
df_clean = {}
df_clean = pd.DataFrame(df_clean)

workbook = pd.read_excel("GDOT_Task3_Data_Scientist_Working_Copy_Aug24_2026.xlsx", sheet_name=None)



Saving GDOT_Task3_Data_Scientist_Working_Copy_Aug24_2026.xlsx to GDOT_Task3_Data_Scientist_Working_Copy_Aug24_2026.xlsx


In [15]:
from numpy import number
class TrackableDataFrame :
  def __init__(self, df):
     self.df_orig = df.copy()
     self.tracking_grid = pd.DataFrame(
          False,
          index=df.index,
          columns=df.columns
          )

  def mark_cell_adjusted(self, row_idx, col_name) :
    if col_name in self.tracking_grid.columns:
      self.tracking_grid.at[row_idx, col_name] = True
  def mark_column_adjusted(self, col_name) :
    if col_name in self.tracking_grid.columns:
      self.tracking_grid[col_name] = True

  def get_manual_review_report(self) :
      report = {}
      for col in self.df_orig.columns:
        col_unaltered = {}
        for row_idx in self.df_orig.index:
          raw_val = self.df_orig.at[row_idx, col]
          if pd.isna(raw_val) or str(raw_val).strip().upper() == 'ND':
                      continue
          if not self.tracking_grid.at[row_idx, col]:
                      col_unaltered[row_idx] = raw_val
        if col_unaltered :
         report[col] = col_unaltered

      return report
def safe_to_string(val):
    # 1. Preserve missing values exactly as they are
    if pd.isna(val):
        return None

    # 2. Preserve 'ND' placeholders
    if str(val).strip().upper() == 'ND':
        return 'ND'

    # 3. Clean up floats that should be integers (e.g., 6.0 -> "6")
    if isinstance(val, float) and val.is_integer():
        return str(int(val))

    # 4. Convert everything else to a string
    return str(val).strip()


# called to change one string into another
# change_from is the original phrase, change_to is the new string
# orig_col is the column you are altering, new_col is the col you are putting the altered version in
# change_every_instance should be set to true if you wish that every instance be replaced rather than replacing if the cell contains only that string
def change_phrase (change_from, change_to, df, col_name, tracker, change_every_instance = False, remove_trailing_space = True) :
  for idx, val in df[col_name].items():
    if pd.isna(val) or str(val).strip().upper() == 'ND':
      continue
    if remove_trailing_space:
      val_str = safe_to_string(val)
    else:
      val_str = str(val)
    if (change_every_instance):
       if change_from in val_str:
         df.at[idx,col_name] = val_str.replace(change_from, change_to)
         tracker.mark_cell_adjusted(idx, col_name)

    else:
      if val_str == change_from:
        df.at[idx,col_name] = change_to
        tracker.mark_cell_adjusted(idx, col_name)
  return df




#be cautious when using if you are unsure if each row has the same number of key symbols/words
#unsplit_data_col_num begins indexing at 1. It is used to place values that did not contain the key word in your choice of colum. column ordering is based on the order you list them
# in new_col_names

def split_col_by_keyword (key, df, tracker, col_name, number_new_col, new_col_names, remove_orig_col = False, unsplit_data_col_num= 1, convert_to_string = False):
  warning = False
  warning2 = False
  if number_new_col != len(new_col_names):
    raise ValueError(f"number_new_col must be equal to the length of new_col_names. Expected '{number_new_col}' names but received '{new_col_names}'")
  if unsplit_data_col_num > (number_new_col):
    raise ValueError(f"unsplit_data_col_num must be equal to or less than the number of columns listed.")
  unsplit_data_col_num = unsplit_data_col_num -1
  for name in range(0, number_new_col, 1):
    tracker.tracking_grid[new_col_names[name]] = False
    tracker.df_orig[new_col_names[name]] = None
    df[new_col_names[name]] = None
  for idx, val in df[col_name].items():
    if pd.isna(val) or str(val).strip().upper() == 'ND':
      tracker.mark_cell_adjusted(idx, col_name)
      for name in new_col_names:
        tracker.mark_cell_adjusted(idx, name)
      continue
    if convert_to_string :
      val = safe_to_string(val)
      df.at[idx, col_name] = val
    if pd.notna(val) and isinstance(val, str) and key in val:
      split_val = val.split(key, maxsplit = number_new_col - 1)
      tracker.mark_cell_adjusted(idx, col_name)

      for index in range(0,number_new_col, 1):
        if index < len(split_val):
          df.at[idx, new_col_names[index]] = split_val[index].strip()
          tracker.mark_cell_adjusted(idx, new_col_names[index])
        else:
          df.at[idx, new_col_names[index]] = None
          tracker.mark_cell_adjusted(idx, new_col_names[index])

          warning = True
    elif isinstance(val, str):
      df.at[idx, new_col_names[unsplit_data_col_num]] = val
    else:
      warning2 = True

  if remove_orig_col:
    df.drop(columns=[col_name], inplace = True)
  if warning:
    print(f"WARNING: At least one of your rows did not contain the number of keywords needed to split it into {number_new_col} columns. The columns are filled in order of how they are listed leaving the remainder blank")
  if warning2:
    num_non_strings = (df[col_name].apply(type) != str).sum()
    print(f" {num_non_strings} row(s) in this column is/are not a string, so the row(s) could not be adjusted")

#This function will split a column that consists of a list into the number of columns in the list.
# It will then mark it as true/false if the category is found in the list for that row
#change exact match = false if you wish to count strings that contain your category as a substring.
#categories are the strings that are an item in your list
def boolean_split(key, df, tracker, col_name, categories, new_col_names, remove_orig_col=False, exact_match=True):
    if len(categories) != len(new_col_names):
        raise ValueError(f"The number of categories ({len(categories)}) must match the number of destination columns ({len(new_col_names)}).")
    warning_unrecognized = False
    unrecognized_items = set()
    cat_set = {cat.strip().lower() for cat in categories}
    for name in new_col_names:
        tracker.tracking_grid[name] = False
        tracker.df_orig[name] = None
        df[name] = None
    for idx, val in df[col_name].items():
        if pd.isna(val) or str(val).strip().upper() == 'ND':
            tracker.mark_cell_adjusted(idx, col_name)
            for name in new_col_names:
                tracker.mark_cell_adjusted(idx, name)
            continue
        if isinstance(val, str):
            tracker.mark_cell_adjusted(idx, col_name)
            val_lower = val.lower()
            split_items = [item.strip().lower() for item in val_lower.split(key)]
            for item in split_items:
                if item and item not in cat_set:
                    unrecognized_items.add(item)
                    warning_unrecognized = True
            for cat, new_col in zip(categories, new_col_names):
                cat_clean = cat.strip().lower()
                if exact_match:
                    is_found = cat_clean in split_items
                else:
                    is_found = cat_clean in val_lower
                if is_found:
                    df.at[idx, new_col] = True
                else:
                    df.at[idx, new_col] = False
                tracker.mark_cell_adjusted(idx, new_col)
    if remove_orig_col:
        df.drop(columns=[col_name], inplace=True)
    if warning_unrecognized:
        print(f"WARNING: Found unrecognized values in '{col_name}' that were not in your categories list: {sorted(list(unrecognized_items))}")

def preview_category_format(df, col_name, allowed_categories):
    series = df[col_name].copy()
    unmatched_counts = {}
    allowed_set = {cat.strip().lower() for cat in allowed_categories}

    for idx, val in series.items():
        if pd.isna(val) or str(val).strip().upper() == 'ND':
            continue
        raw_str = str(val).strip()
        cleaned_str = " ".join(raw_str.split())
        if cleaned_str.lower() in allowed_set:
            continue
        unmatched_counts[raw_str] = unmatched_counts.get(raw_str, 0) + 1

    if not unmatched_counts:
        print(f"All values in '{col_name}' are uniform and match your allowed categories!")
        return None

    report_data = []
    for raw_val, count in unmatched_counts.items():
        cleaned_str = " ".join(raw_val.strip().split()).lower()
        suggestions = []
        for cat in allowed_categories:
            cat_clean = cat.strip()
            cat_lower = cat_clean.lower()

            has_substring_match = False
            min_len = min(4, len(cat_lower))
            for i in range(len(cat_lower) - min_len + 1):
                sub = cat_lower[i:i+min_len]
                if sub in cleaned_str:
                    has_substring_match = True
                    break
            if has_substring_match:
                suggestions.append(cat_clean)

        if suggestions:
            status = f"Possible Alternative: {', '.join(suggestions)}"
        else:
            status = "Unknown"

        report_data.append({
            "Raw Value": raw_val,
            "Count": count,
            "Suggestion/Status": status
        })

    report_df = pd.DataFrame(report_data)
    report_df = report_df.sort_values(by="Count", ascending=False).reset_index(drop=True)

    print(f"\n=== CONSOLIDATED AUDIT REPORT FOR '{col_name}' ===")
    print(report_df.to_string(index=False))
    print("================================================\n")

def safe_numeric_conversion(df, col_name, tracker):

#only use if number is without any characters besides numerics, commas, percentages and decimals
  def percent_format(df, col_name, tracker):
    for idx, val in df[col_name].items():
      if pd.isna(val) or str(val).strip().upper() == 'ND':
        continue
      if not (isinstance(val,(int,float))):
        if not (isinstance(val,(str))):
          print(f"Warning: cell at row {idx + 1} in column {col_name} is not a numeric value or a string & cannot be adjusted")
          continue
        if ("," in val) or ("%" in val):
          tracker.mark_cell_adjusted(idx,col_name)
          df.at[idx, col_name] = df.at[idx, col_name].replace(",", "")
      try:
        if not (isinstance(val, (int,float))):

          if("%" in val):
            df.at[idx, col_name] = df.at[idx, col_name].replace("%", "")
            df.at[idx, col_name] = float(df.at[idx,col_name])
            df.at[idx, col_name] = (df.at[idx, col_name]/100)
          else:
            df.at[idx, col_name] = float(df.at[idx, col_name])
      except ValueError:
        print(f"Warning: cell at row {idx + 1} in {col_name} failed numeric conversion")

def convert_column_to_numeric(df, col_name):
    """
    Loops through a column, safely skips blank and 'ND' values,
    and tries to convert the rest to a numeric float or integer.
    """
    for idx, val in df[col_name].items():
        # 1. Skip blanks and 'ND' placeholders (Missing Data Rule)
        if pd.isna(val) or str(val).strip().upper() == 'ND':
            continue

        # 2. Try to convert the cell value to a number
        try:
            num_val = float(val)

            # Clean whole decimals to integers (e.g., 1200.0 -> 1200)
            if num_val.is_integer():
                df.at[idx, col_name] = int(num_val)
            else:
                df.at[idx, col_name] = num_val

        except ValueError:
            # 3. Catch conversion failures (like "1–3 ft" or "Fine; Medium")
            # Prints warning with Excel-friendly 1-based index (idx + 1)
            print(f"Warning: cell at row {idx + 1} in '{col_name}' failed numeric conversion. Raw value: '{val}'")

    return df
def remove_character(chars_to_remove, df, col_name, tracker):
  for idx, val in df[col_name].items():
    modified = False
    if pd.isna(val) or str(val).strip().upper() == 'ND':
      continue
    val_str = str(val)

    for char in chars_to_remove:
      if char in val_str:
        val_str = val_str.replace(char, "")
        df.at[idx, col_name] = val_str
        modified = True
    if modified:
      tracker.mark_cell_adjusted(idx, col_name)
  return df


def code_string_vals (original_vals, coded_vals, df, col_name, tracker, code_unlisted_values = False) :
  if len(original_vals) != len(coded_vals):
    raise ValueError("Length of possible values & coded values must match.")
  for idx, val in df[col_name].items():
    modified = False
    if pd.isna(val) or str(val).strip().upper() == 'ND':
      continue
    val = str(val)
    for count, orig in enumerate(original_vals, start = 0):
      if (orig in val):
        df.at[idx, col_name] = str(df.at[idx, col_name])
        df.at[idx, col_name] = df.at[idx, col_name].replace(orig, coded_vals[count])
        modified = True
    if code_unlisted_values and (not modified):
        df.at[idx,col_name] = "unknown"
        modified = True
    if modified:
      tracker.mark_cell_adjusted(idx, col_name)



















In [ ]:
site = workbook["Site_Adaptation"]
site_clean = site.copy()
tracker = TrackableDataFrame(site_clean)
split_col_by_keyword(";", site_clean, tracker,"Georgia_Hardiness_Zone",2, ["Georgia_Zone", "US_Zone"], False, 1, True)
tracker.tracking_grid["Georgia_Hardiness_Zone"]
site_clean['Georgia_Zone']
#tracker.get_manual_review_report()


,Georgia_Zone
0,All of Georgia
1,All of Georgia
2,All of Georgia
3,All of Georgia
4,All of Georgia
...,...
219,All of Georgia
220,All of Georgia
221,All of Georgia
222,7–8
